In [7]:
# =============================================================================
# FUNCTION CALLING & MCP
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHERE ARE WE ON THE PATH?
# ---------------------------------------------------------------------------
#   Pretrain → SFT → DPO → Quantization  = teach + shrink the model
#   THIS NOTEBOOK                        = give the model HANDS
#
# Easy analogy:
#   A language model alone is a very smart brain in a locked room.
#   It can talk about prices, but it cannot LOOK UP a live price.
#   Function calling = give it a phone to call your code.
#   MCP (Model Context Protocol) = a standard "phone book + dial tone"
#   so many apps can expose tools the same way (more on that later).
#
# Why care for your earpiece product?
#   On a live sales call you need real facts: plan price, CRM discount,
#   calendar, docs — not guesses. The model decides WHEN to call a tool;
#   YOUR Python code does the real lookup and returns the answer.
#
#
# ---------------------------------------------------------------------------
# THE BIG PICTURE (agent loop)
# ---------------------------------------------------------------------------
#   1) User asks something  ("What's the starter plan price?")
#   2) Prompt includes TOOL DESCRIPTIONS (this cell builds those tools)
#   3) Model either answers in plain text OR outputs a JSON function_call
#   4) Your code parses JSON → runs the real Python function
#   5) You feed the tool RESULT back to the model → final spoken answer
#
# This cell = step 2's ingredients: real functions + a schema the model reads.
#
#
# ---------------------------------------------------------------------------
# PARTS A–C live in week2/sales_tools.py (importable from any cell)
# ---------------------------------------------------------------------------
# Same mock DBs, callables, TOOL_IMPL map, and JSON `tools` schemas as before —
# extracted so later cells do not depend on "run cell 0 first".

from pathlib import Path
import sys

_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

from sales_tools import (
    PRODUCT_DB,
    CUSTOMER_DB,
    get_product_price,
    get_customer_discount,
    TOOL_IMPL,
    tools,
)

print("Tools ready:", [t["name"] for t in tools])
print(get_product_price("starter"))
print(get_customer_discount("acme_corp"))


Tools ready: ['get_product_price', 'get_customer_discount']
The starter plan costs $99 per month.
acme_corp is on the enterprise tier with a 15.0% discount.


In [ ]:
# =============================================================================
# PROMPT THE MODEL TO USE TOOLS
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHAT THIS CELL DOES (picture)
# ---------------------------------------------------------------------------
# The model cannot see your Python functions. It only sees TEXT.
# So we BUILD a prompt that includes:
#   1) role ("AI sales coach")
#   2) the TOOL SCHEMAS (names, descriptions, argument shapes)
#   3) the JSON shape to use when requesting a call
#   4) the live context + user question
#
#   tools (JSON schemas)
#        │
#        ▼
#   ┌─────────────────────────────────────────┐
#   │  PROMPT string                          │
#   │  ... Available tools: [...]             │
#   │  ... If you need a tool, reply JSON ... │
#   │  ... User: What's the starter price?    │
#   │  Assistant:                             │
#   └─────────────────────────────────────────┘
#        │
#        ▼  (later) model generate()
#   either plain text OR {"function_call": {...}}
#
# Sticky: schemas = the "menu"; function_call JSON = the "order ticket".
# YOUR code (next cells) cooks the order by running real Python.
#

import json
from pathlib import Path
import sys

# Make week2/ importable whether cwd is repo root or week2/
_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

# Shared mock backends + JSON tool registry (week2/sales_tools.py)
from sales_tools import (
    PRODUCT_DB,
    CUSTOMER_DB,
    get_product_price,
    get_customer_discount,
    TOOL_IMPL,
    tools,  # list of {name, description, parameters} — what goes IN the prompt
)


def build_prompt_with_tools(ground_truth, user_utterance, tools):
    """Build an instruction prompt that lists tools and the call format.

    ground_truth   : optional context (call notes, product facts, etc.)
    user_utterance : what the human just said
    tools          : schema list from sales_tools (NOT the Python callables)
    """
    # Pretty-print schemas so the model can read names + required args.
    tool_desc = json.dumps(tools, indent=2)

    # Shown to the model as the ONLY allowed call shape.
    # (Kept outside the f-string so we do not fight with { } escaping.)
    example_call = """
{
  "function_call": {
    "name": "<function_name>",
    "arguments": {"arg_name": "value"}
  }
}
""".strip()

    # f-string fills ground_truth / user_utterance / tool_desc / example_call.
    # Tip: if you put literal JSON braces inside an f-string, double them: {{ }}
    prompt = f"""You are an AI sales coach. You can call external functions to retrieve real-time data.

Available tools (JSON):
{tool_desc}

If you need a tool, reply with ONLY a JSON object in this exact shape:
{example_call}

Rules:
- Use a tool when you need a live fact (price, discount, etc.).
- If you do not need a tool, reply with a normal helpful answer (plain text, no JSON).
- Never invent tool results — only request a call; the system will run it.

Context / ground truth: {ground_truth}
User: {user_utterance}
Assistant:"""
    return prompt


# Demo: a question that SHOULD need get_product_price (live fact, not memorized chat).
demo_prompt = build_prompt_with_tools(
    ground_truth="Live sales call. Customer asked about pricing.",
    user_utterance="What's the monthly price of the starter plan?",
    tools=tools,
)
print(demo_prompt)
print("\n--- prompt length:", len(demo_prompt), "chars ---")
# Expect: tool schemas in the middle, user question at the bottom, ends with "Assistant:"
# Next cell: parse a fake model JSON reply and run TOOL_IMPL[name](**arguments).


You are an AI sales coach. You can call external functions to retrieve real-time data.

Available tools (JSON):
[
  {
    "name": "get_product_price",
    "description": "Get the monthly price of a product plan.",
    "parameters": {
      "type": "object",
      "properties": {
        "product_name": {
          "type": "string",
          "description": "Plan name (starter, professional, enterprise)"
        }
      },
      "required": [
        "product_name"
      ]
    }
  },
  {
    "name": "get_customer_discount",
    "description": "Get the discount and tier for a customer company.",
    "parameters": {
      "type": "object",
      "properties": {
        "company": {
          "type": "string",
          "description": "Company name (acme_corp, startup_inc)"
        }
      },
      "required": [
        "company"
      ]
    }
  }
]

If you need a tool, reply with ONLY a JSON object in this exact shape:
{
  "function_call": {
    "name": "<function_name>",
    "arguments":

In [9]:
# =============================================================================
# PARSE + EXECUTE (the middle of the agent loop)
# =============================================================================
#
# The model does not execute tools. YOUR code does:
#   model text  →  find JSON  →  read name + arguments  →  TOOL_IMPL[name](**args)
#
# Below we fake a model reply (as if the LLM already chose a tool),
# then run the same path you would after a real generate() call.
#

import re
import json

from pathlib import Path
import sys

_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

from sales_tools import (
    PRODUCT_DB,
    CUSTOMER_DB,
    get_product_price,
    get_customer_discount,
    TOOL_IMPL,
    tools,
)



def extract_function_call(model_text: str):
    """Return {"name": ..., "arguments": {...}} or None if plain text."""
    # Find the first {...} JSON object in the reply (models sometimes add chatter).
    match = re.search(r"\{[\s\S]*\}", model_text)
    if not match:
        return None
    try:
        payload = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

    # Support either {"function_call": {...}} or a bare {"name", "arguments"}.
    call = payload.get("function_call", payload)
    if not isinstance(call, dict) or "name" not in call:
        return None
    args = call.get("arguments", {})
    if not isinstance(args, dict):
        return None
    return {"name": call["name"], "arguments": args}


def run_tool(call: dict) -> str:
    """Dispatch one parsed call through TOOL_IMPL."""
    name = call["name"]
    args = call["arguments"]
    if name not in TOOL_IMPL:
        return f"Error: unknown tool '{name}'."
    try:
        return TOOL_IMPL[name](**args)
    except TypeError as e:
        return f"Error: bad arguments for '{name}': {e}"


def handle_model_reply(model_text: str) -> str:
    """If the model requested a tool, run it; else return the text as-is."""
    call = extract_function_call(model_text)
    if call is None:
        return model_text.strip()
    print("Model requested tool:", call)
    result = run_tool(call)
    print("Tool result:", result)
    # In a full agent loop you would send `result` back into another prompt
    # so the model can phrase a natural final answer for the earpiece.
    return result


# --- Simulated model outputs (pretend generate() returned these) ---
fake_tool_reply = """
{
  "function_call": {
    "name": "get_product_price",
    "arguments": {"product_name": "starter"}
  }
}
"""

fake_plain_reply = "Sounds good — happy to walk through pricing whenever you're ready."

print("=== case: model wants a tool ===")
print("Final:", handle_model_reply(fake_tool_reply))
print()
print("=== case: model answers directly ===")
print("Final:", handle_model_reply(fake_plain_reply))


=== case: model wants a tool ===
Model requested tool: {'name': 'get_product_price', 'arguments': {'product_name': 'starter'}}
Tool result: The starter plan costs $99 per month.
Final: The starter plan costs $99 per month.

=== case: model answers directly ===
Final: Sounds good — happy to walk through pricing whenever you're ready.


In [10]:
# =============================================================================
# HOW THIS CONNECTS TO MCP (Model Context Protocol)
# =============================================================================
#
# What you built above is "DIY function calling":
#   - You wrote tool schemas by hand
#   - You stuffed them into a prompt
#   - You parsed JSON and called Python yourself
#
# That pattern is the CORE idea behind every tool-using agent.
#
# MCP standardizes the same idea across apps:
#
#   ┌─────────────┐         ┌──────────────────┐         ┌─────────────────┐
#   │ Host / IDE  │ ◄─────► │ MCP Client        │ ◄─────► │ MCP Server      │
#   │ (Cursor)    │         │ (talks protocol)  │         │ (exposes tools) │
#   └─────────────┘         └──────────────────┘         └─────────────────┘
#
# Easy mapping to THIS notebook:
#   tools list     ≈  what an MCP server advertises via tools/list
#   function_call  ≈  tools/call  { name, arguments }
#   TOOL_IMPL[...] ≈  the server's real handler (DB, API, file, …)
#   tool result    ≈  tools/call response content sent back to the model
#
# Why MCP exists (one sentence):
#   So every editor / agent does not invent a private tool format —
#   servers expose tools once; many clients can use them.
#
# For your earpiece copilot later:
#   CRM lookup, calendar, playbook search can each be an MCP server (or
#   plain functions first). Start with the DIY loop above; swap the
#   transport for MCP when you want reusable, shareable tools.
#
# Mental model:
#   Function calling = the skill (model asks → code runs → result returns)
#   MCP              = a shared socket + catalog for that skill
#

print("DIY function calling: DONE (schemas + prompt + parse + execute)")
print("MCP: same loop, standardized client/server transport")
print()
print("Next practice: change user_utterance to ask about acme_corp discount,")
print("fake a get_customer_discount call, and run handle_model_reply(...).")


DIY function calling: DONE (schemas + prompt + parse + execute)
MCP: same loop, standardized client/server transport

Next practice: change user_utterance to ask about acme_corp discount,
fake a get_customer_discount call, and run handle_model_reply(...).


In [13]:
# =============================================================================
# MCP SERVER (IN-PROCESS) — same tools, protocol-shaped requests
# =============================================================================
#
# Picture:
#   DIY loop (cells above): you call Python functions yourself.
#   MCP shape: send a JSON request string → server.handle_request → JSON reply.
#
#   Client:  {"method": "tools/list"}
#   Server:  {"tools": [ {...schemas...} ]}
#
#   Client:  {"method": "tools/call", "params": {"name": "...", "arguments": {...}}}
#   Server:  {"content": [{"type": "text", "text": "...result..."}]}
#
# Production MCP often runs as a separate process over stdio/HTTP.
# Here the "server" is just a Python class in the same notebook — easier to learn.
#
# Requires earlier cells: get_product_price, get_customer_discount, and `tools`
# (schemas). Re-run from the top if those names are missing.
#

import json

from pathlib import Path
import sys

_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

from sales_tools import (
    PRODUCT_DB,
    CUSTOMER_DB,
    get_product_price,
    get_customer_discount,
    TOOL_IMPL,
    tools,
)


class MCPServer:
    """Tiny in-process stand-in for an MCP tools server."""

    def __init__(self, tool_impl: dict, tool_schemas: list):
        # name → Python callable (the real work)
        self.tool_impl = dict(tool_impl)
        # JSON schemas advertised to clients / models (NOT the callables)
        self.tool_schemas = list(tool_schemas)

    def handle_request(self, request_json: str) -> str:
        """Handle one JSON request; always return a JSON string."""
        try:
            req = json.loads(request_json)
            method = req.get("method")

            # Catalog: what can I call?
            if method == "tools/list":
                return json.dumps({"tools": self.tool_schemas})

            # Execute: run one registered function
            if method == "tools/call":
                params = req["params"]
                func_name = params["name"]
                arguments = params.get("arguments", {})
                if func_name not in self.tool_impl:
                    return json.dumps({"error": f"Tool not found: {func_name}"})
                result = self.tool_impl[func_name](**arguments)
                # MCP-ish content wrapper (text payload)
                return json.dumps({
                    "content": [{"type": "text", "text": result}]
                })

            return json.dumps({"error": f"Unknown method: {method}"})
        except Exception as e:
            return json.dumps({"error": str(e)})


# Wire shared backends + schemas (week2/sales_tools.py)
server = MCPServer(tool_impl=TOOL_IMPL, tool_schemas=tools)

# --- Verify both MCP-style methods ---
list_reply = server.handle_request(json.dumps({"method": "tools/list"}))
print("tools/list →", list_reply[:120], "...")

call_reply = server.handle_request(json.dumps({
    "method": "tools/call",
    "params": {
        "name": "get_product_price",
        "arguments": {"product_name": "professional"},
    },
}))
print("tools/call →", call_reply)


tools/list → {"tools": [{"name": "get_product_price", "description": "Get the monthly price of a product plan.", "parameters": {"type ...
tools/call → {"content": [{"type": "text", "text": "The professional plan costs $299 per month."}]}


In [17]:
# =============================================================================
# MCP CLIENT — thin wrapper that talks JSON to the MCP server
# =============================================================================
#
# Picture:
#   YOU / agent code          MCPClient              MCPServer
#   call_tool("price", ...) → tools/call JSON  →  runs Python → JSON result
#   list_tools()            → tools/list JSON  →  returns schemas
#
# Client does NOT contain business logic. It only packages requests.
# Server (previous cell) owns the real functions.
#

import json

from pathlib import Path
import sys

_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

from sales_tools import TOOL_IMPL, tools


# Reuse server from previous cell, or build one if you jumped here directly
if "server" not in globals():
    from sales_tools import TOOL_IMPL, tools as _schemas

    class _MCPServer:
        def __init__(self, tool_impl, tool_schemas):
            self.tool_impl = dict(tool_impl)
            self.tool_schemas = list(tool_schemas)

        def handle_request(self, request_json: str) -> str:
            req = json.loads(request_json)
            method = req.get("method")
            if method == "tools/list":
                return json.dumps({"tools": self.tool_schemas})
            if method == "tools/call":
                params = req["params"]
                name, arguments = params["name"], params.get("arguments", {})
                if name not in self.tool_impl:
                    return json.dumps({"error": f"Tool not found: {name}"})
                text = self.tool_impl[name](**arguments)
                return json.dumps({"content": [{"type": "text", "text": text}]})
            return json.dumps({"error": f"Unknown method: {method}"})

    server = _MCPServer(TOOL_IMPL, _schemas)


class MCPClient:
    """Speak MCP-shaped JSON to an in-process server."""

    def __init__(self, server):
        self.server = server

    def call_tool(self, name, arguments):
        # Build tools/call request → server runs the function → parse reply
        request = json.dumps({
            "method": "tools/call",
            "params": {"name": name, "arguments": arguments},
        })
        response = self.server.handle_request(request)
        return json.loads(response)

    def list_tools(self):
        request = json.dumps({"method": "tools/list"})
        return json.loads(self.server.handle_request(request))


client = MCPClient(server)

# Quick smoke test
print("Listed:", [t["name"] for t in client.list_tools()["tools"]])
resp = client.call_tool("get_product_price", {"product_name": "professional"})
print("call_tool →", resp)
# Expect content[0].text ≈ "The professional plan costs $299 per month."


Listed: ['get_product_price', 'get_customer_discount']
call_tool → {'content': [{'type': 'text', 'text': 'The professional plan costs $299 per month.'}]}


In [18]:
# =============================================================================
# ASSISTANT + TOOLS LOOP (function calling end-to-end)
# =============================================================================
#
# Full loop picture:
#
#   1) build_prompt_with_tools(...)     # schemas + user question
#   2) model.generate → text            # may be JSON function_call OR plain answer
#   3) parse_function_call(text)
#        ├─ found → client.call_tool(...) → append Function result
#        │          → generate AGAIN → final spoken/correction answer
#        └─ none  → return the plain text answer
#
# This notebook may not have a trained policy_model from SFT/DPO.
# So we demo with a SIMULATED first model reply (realistic JSON call),
# which still exercises: parse → MCP client → second-pass final answer.
#

import json
import re
from pathlib import Path
import sys

_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

from sales_tools import TOOL_IMPL, tools

# Need prompt builder from the PROMPT THE MODEL cell
if "build_prompt_with_tools" not in globals():
    raise NameError("Run the prompt cell (build_prompt_with_tools) before this one.")

# Need MCP client from the previous cell
if "client" not in globals():
    raise NameError("Run the MCPClient cell before this one.")


def parse_function_call(text):
    """Extract {name, arguments} from model text (fenced JSON or bare JSON)."""
    # 1) ```json ... ``` fence (some models wrap)
    match = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    candidates = [match.group(1)] if match else []
    # 2) first {...} object anywhere (matches our prompt's ONLY JSON rule)
    bare = re.search(r"\{[\s\S]*\}", text)
    if bare:
        candidates.append(bare.group(0))

    for raw in candidates:
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            continue
        call = data.get("function_call", data)
        if isinstance(call, dict) and "name" in call:
            args = call.get("arguments", {})
            if isinstance(args, dict):
                return {"name": call["name"], "arguments": args}
    return None


def _generate_text(model, tokenizer, prompt, max_new_tokens=50, temperature=0.7):
    """Real MiniGPT path if model+tokenizer exist; else None (use simulation)."""
    if model is None or tokenizer is None:
        return None
    import torch

    device_ = globals().get("device", "cpu")
    ids = tokenizer.encode(prompt)
    input_ids = torch.tensor([ids], dtype=torch.long).to(device_)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids, max_new_tokens=max_new_tokens, temperature=temperature
        )
    return tokenizer.decode(output_ids[0].tolist())


def run_assistant_with_tools(
    ground_truth,
    user_utterance,
    model=None,
    tokenizer=None,
    simulated_first_reply=None,
):
    """One agent turn: prompt → (optional tool call via MCP) → final answer."""
    prompt = build_prompt_with_tools(ground_truth, user_utterance, tools)

    # --- first model reply ---
    response_text = _generate_text(model, tokenizer, prompt, max_new_tokens=50)
    if response_text is None:
        # Simulation: pretend the LLM requested a live price lookup
        response_text = simulated_first_reply or json.dumps(
            {
                "function_call": {
                    "name": "get_product_price",
                    "arguments": {"product_name": "professional"},
                }
            }
        )
        print("[sim] first model reply:", response_text)

    func_call = parse_function_call(response_text)
    if not func_call:
        if "### Response:" in response_text:
            return response_text.split("### Response:")[-1].strip()
        return response_text.strip()

    # --- tool path via MCP client ---
    print("Parsed function_call:", func_call)
    tool_result = client.call_tool(func_call["name"], func_call["arguments"])
    result_text = tool_result.get("content", [{}])[0].get(
        "text", tool_result.get("error", "Error")
    )
    print("Tool result:", result_text)

    new_prompt = (
        prompt
        + "\nFunction result: "
        + result_text
        + "\nNow provide the correction / final answer for the user:"
    )

    final_answer = _generate_text(model, tokenizer, new_prompt, max_new_tokens=40)
    if final_answer is None:
        final_answer = "Correction: " + result_text
        print("[sim] final answer:", final_answer)
        return final_answer

    marker = "Now provide the correction"
    if marker in final_answer:
        return final_answer.split(marker)[-1].strip(" :\n")
    return final_answer.strip()


# Demo — works WITHOUT policy_model (simulated LLM JSON + real MCP tools)
answer = run_assistant_with_tools(
    ground_truth="Live sales call. Confirm live catalog price for Professional.",
    user_utterance="What's the price of Professional?",
    model=globals().get("policy_model"),
    tokenizer=globals().get("tokenizer"),
)
print("")
print("=== Assistant final ===")
print(answer)


[sim] first model reply: {"function_call": {"name": "get_product_price", "arguments": {"product_name": "professional"}}}
Parsed function_call: {'name': 'get_product_price', 'arguments': {'product_name': 'professional'}}
Tool result: The professional plan costs $299 per month.
[sim] final answer: Correction: The professional plan costs $299 per month.

=== Assistant final ===
Correction: The professional plan costs $299 per month.
